
# TP 3 — MLP & optimisation structurée (`nn.Module` + `Trainer`)

**Objectifs de la séance :**
- Passer de paramètres manuels (`w`, `b`) à un modèle `nn.Module`.
- Étendre le modèle linéaire en un véritable MLP (couche cachée + activation).
- Construire progressivement la classe `Trainer` que vous réutiliserez jusqu'à la fin du cours.
- Comparer expérimentalement SGD, SGD+momentum et Adam sur un même problème.

On repart du même jeu de données que la séance 2 (régression linéaire univariée synthétique), pour se concentrer sur l'outillage plutôt que sur un nouveau problème.

In [ ]:

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(0)


## Jeu de données (identique à la séance 2)

In [ ]:

X = torch.rand(100, 1)
# w* = -3, b* = 1.5
y = -3.0 * X + 1.5 + 0.4 * torch.randn(X.size())

dataset = TensorDataset(X, y)
train_loader = DataLoader(dataset, batch_size=10, shuffle=True)

plt.scatter(X.numpy(), y.numpy())
plt.xlabel("X"); plt.ylabel("y")


## Partie 1 — De `w`, `b` à `nn.Module`

Jusqu'ici, `w` et `b` étaient deux tenseurs isolés. `nn.Module` est la classe de base de PyTorch pour définir un modèle : elle regroupe les paramètres, sait les lister (`.parameters()`), et définit le calcul à effectuer dans une méthode `forward`.

**Question 1.1.** Complétez la classe `LinearRegression` ci-dessous : dans `__init__`, définissez une couche `nn.Linear(in_features=1, out_features=1)` (elle contient exactement un poids et un biais, comme votre `w`/`b` manuels) ; dans `forward`, appliquez cette couche à `x` et renvoyez le résultat.

In [ ]:
class LinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)  # TODO : définir self.linear

    def forward(self, x):
        return self.linear(x)  # TODO : appliquer self.linear à x et renvoyer le résultat


model = LinearRegression()
print(model)
print("Nombre de paramètres :", sum(p.numel() for p in model.parameters()))


## Partie 2 — Une boucle d'entraînement générique

**Question 2.1.** Écrivez une fonction `train_one_epoch(model, loader, optimizer, loss_fn)` qui parcourt `loader`, et pour chaque mini-batch `(xb, yb)` : calcule les prédictions (`preds = model(xb)`), la perte (`loss_fn(preds, yb)`), effectue `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()`, et renvoie la perte moyenne sur l'epoch (pondérée par la taille de chaque mini-batch, `xb.size(0)`).

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn):
    total_loss, n = 0.0, 0
    for xb, yb in loader:
        # TODO : forward, perte, zero_grad, backward, step
        preds = model(xb)
        loss = loss_fn(preds, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        n += xb.size(0)
    return total_loss / n


**Question 2.2.** Utilisez `train_one_epoch` pour entraîner `model` pendant 50 epochs, avec `torch.optim.SGD(model.parameters(), lr=0.1)` et `loss_fn = nn.MSELoss()`. Stockez la perte de chaque epoch et tracez la courbe. Retrouvez-vous des valeurs de poids proches de $w^\star=-3$, $b^\star=1.5$ (regardez `model.linear.weight` et `model.linear.bias`) ?

In [ ]:
# TODO : boucle sur 50 epochs appelant train_one_epoch, stockage des pertes dans une liste,
#        puis tracé de la courbe et affichage de model.linear.weight / model.linear.bias
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

losses_lin = []
for epoch in range(50):
    epoch_loss = train_one_epoch(model, train_loader, optimizer, loss_fn)
    losses_lin.append(epoch_loss)

plt.plot(losses_lin)
plt.xlabel("epoch"); plt.ylabel("MSE")
plt.title("LinearRegression - train_one_epoch")
plt.show()

print("weight :", model.linear.weight.item(), " (attendu : -3)")
print("bias   :", model.linear.bias.item(), " (attendu : 1.5)")


## Partie 3 — Un vrai MLP

`LinearRegression` n'est qu'une seule couche : c'est encore une régression linéaire. Un MLP empile plusieurs couches `nn.Linear`, séparées par des fonctions d'activation non linéaires.

**Question 3.1.** Complétez `MLP` : une couche cachée suivie d'une activation `nn.ReLU()`, puis une couche de sortie.

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_size=16):
        super().__init__()
        # TODO : définir self.hidden = nn.Linear(1, hidden_size)
        #        self.activation = nn.ReLU()
        #        self.output = nn.Linear(hidden_size, 1)
        self.hidden = nn.Linear(1, hidden_size)
        self.activation = nn.ReLU()
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # TODO : enchaîner hidden -> activation -> output
        x = self.hidden(x)
        x = self.activation(x)
        return self.output(x)


mlp = MLP(hidden_size=16)
print(mlp)
print("Nombre de paramètres :", sum(p.numel() for p in mlp.parameters()))


**Question 3.2.** Entraînez `mlp` avec `train_one_epoch`, de la même façon qu'en partie 2 (mêmes hyperparamètres). Le problème étant en réalité linéaire, le MLP n'a ici aucune raison de faire mieux que `LinearRegression` — c'est normal, l'intérêt du MLP apparaîtra sur des problèmes non linéaires dans les séances suivantes. Vérifiez simplement que l'entraînement converge.

In [ ]:
# TODO : entraîner mlp pendant 50 epochs
#        et tracer la courbe de perte
optimizer = torch.optim.SGD(mlp.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

losses_mlp = []
for epoch in range(50):
    epoch_loss = train_one_epoch(mlp, train_loader, optimizer, loss_fn)
    losses_mlp.append(epoch_loss)

plt.plot(losses_mlp)
plt.xlabel("epoch"); plt.ylabel("MSE")
plt.title("MLP - train_one_epoch")
plt.show()


## Partie 4 — Construire le `Trainer`

`train_one_epoch` ne gère pas l'évaluation sur un jeu de validation, ni le suivi d'un historique. Plutôt que de dupliquer du code à chaque séance, on va encapsuler cette logique dans une classe réutilisable.

**Question 4.1.** Complétez la classe `SimpleTrainer` ci-dessous : la méthode `_run_epoch` doit parcourir `loader`, calculer prédictions et perte pour chaque mini-batch, et **si `train=True`** effectuer la rétropropagation et la mise à jour des poids (si `train=False`, aucune mise à jour : c'est le mode évaluation). Elle doit renvoyer la perte moyenne sur l'epoch.

In [ ]:
class SimpleTrainer:
    def __init__(self, model, optimizer, loss_fn):
        self.model = model
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.history = {"train_loss": [], "val_loss": []}

    def _run_epoch(self, loader, train):
        self.model.train(train)
        total_loss, n = 0.0, 0
        with torch.set_grad_enabled(train):
            for xb, yb in loader:
                # TODO : calculer preds et loss
                preds = self.model(xb)
                loss = self.loss_fn(preds, yb)

                # TODO : si train est True, faire optimizer.zero_grad() / loss.backward() / optimizer.step()
                if train:
                    self.optimizer.zero_grad()
                    loss.backward()
                    self.optimizer.step()

                total_loss += loss.item() * xb.size(0)
                n += xb.size(0)
        return total_loss / n

    def fit(self, train_loader, val_loader=None, epochs=10, verbose=True):
        for epoch in range(epochs):
            train_loss = self._run_epoch(train_loader, train=True)
            self.history["train_loss"].append(train_loss)
            msg = f"Epoch {epoch + 1}/{epochs} - train_loss: {train_loss:.4f}"

            if val_loader is not None:
                val_loss = self._run_epoch(val_loader, train=False)
                self.history["val_loss"].append(val_loss)
                msg += f" - val_loss: {val_loss:.4f}"

            if verbose:
                print(msg)
        return self.history

    def evaluate(self, loader):
        return self._run_epoch(loader, train=False)


**Question 4.2.** Réentraînez `mlp` (recréez une instance neuve, pour repartir d'une initialisation aléatoire) avec `SimpleTrainer` plutôt qu'avec `train_one_epoch`. Vous devriez obtenir une courbe de perte similaire à celle de la partie 3.

In [ ]:
# TODO : recréer un MLP neuf, un optimizer, un SimpleTrainer, et appeler .fit(train_loader, epochs=50)
mlp_new = MLP(hidden_size=16)
optimizer = torch.optim.SGD(mlp_new.parameters(), lr=0.1)
simple_trainer = SimpleTrainer(mlp_new, optimizer, nn.MSELoss())
history = simple_trainer.fit(train_loader, epochs=50)

plt.plot(history["train_loss"])
plt.xlabel("epoch"); plt.ylabel("MSE")
plt.title("MLP - SimpleTrainer")
plt.show()


Ce que vous venez d'écrire, c'est (une version simplifiée de) la classe `Trainer` fournie dans `training_toolbox.py` : mêmes idées (`fit`/`evaluate`, `_run_epoch`), avec en plus la gestion de métriques et de callbacks que l'on découvrira dans les séances suivantes, sans jamais casser ce que vous venez d'écrire. À partir de maintenant, on utilise directement cette version « officielle ».

In [ ]:

from training_toolbox import Trainer

mlp_demo = MLP(hidden_size=16)
trainer_demo = Trainer(mlp_demo, torch.optim.Adam(mlp_demo.parameters(), lr=0.05), nn.MSELoss())
trainer_demo.fit(train_loader, epochs=10)


## Partie 5 — SGD vs SGD+momentum vs Adam

**Question 5.1.** En utilisant le `Trainer` de `training_toolbox`, entraînez trois MLP **neufs** (même architecture, `hidden_size=16`), avec respectivement :

- `torch.optim.SGD(params, lr=0.1)` ;
- `torch.optim.SGD(params, lr=0.1, momentum=0.9)` ;
- `torch.optim.Adam(params, lr=0.1)`.

Entraînez chacun pendant 50 epochs sur `train_loader`, en stockant l'historique de perte (`history["train_loss"]`), puis tracez les trois courbes sur un même graphique.

In [ ]:
# TODO : entraîner trois MLP neufs (hidden_size=16) avec SGD, SGD+momentum et Adam
#        (mêmes 50 epochs, mêmes données) et tracer les trois courbes de train_loss
optim_factories = {
    "SGD": lambda params: torch.optim.SGD(params, lr=0.1),
    "SGD+momentum": lambda params: torch.optim.SGD(params, lr=0.1, momentum=0.9),
    "Adam": lambda params: torch.optim.Adam(params, lr=0.1),
}

results = {}
for name, make_optimizer in optim_factories.items():
    model_i = MLP(hidden_size=16)
    optimizer_i = make_optimizer(model_i.parameters())
    trainer_i = Trainer(model_i, optimizer_i, nn.MSELoss())
    history_i = trainer_i.fit(train_loader, epochs=50, verbose=False)
    results[name] = history_i["train_loss"]

for name, losses in results.items():
    plt.plot(losses, label=name)
plt.xlabel("epoch"); plt.ylabel("MSE"); plt.legend()
plt.title("SGD vs SGD+momentum vs Adam")
plt.show()


**Questions.**
- Quel optimiseur converge le plus vite ici ? Le plus stable (courbe la moins bruitée) ?
- Le momentum change-t-il beaucoup les choses par rapport à SGD seul, sur ce problème très simple ? Sur quel type de paysage de perte son intérêt serait-il plus visible ?

_Sur ce problème très simple (régression linéaire, paysage convexe et bien conditionné), Adam converge généralement le plus vite ici ; SGD+momentum est un peu plus rapide et un peu plus stable que SGD seul, mais l'écart reste modeste. Le momentum apporte surtout un gain net sur des paysages de perte mal conditionnés (vallées étroites, ravins) ou en présence de bruit de gradient important — des situations bien plus fréquentes avec des architectures et des jeux de données plus complexes (CNN, etc.) que sur cette régression linéaire jouet._


## Bilan

Vous savez maintenant définir un modèle avec `nn.Module`, construire (et donc comprendre de l'intérieur) un `Trainer`, et comparer des optimiseurs. À partir de la séance suivante, vous utiliserez systématiquement `from training_toolbox import Trainer` sans le reconstruire.